# Perfilado de la union de archivos canonicos

Revisión de los registros antes de la limpieza. Se generan tablas y gráficas de diagnóstico, sin modificar el CSV de entrada.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from html import unescape

archivo = "../02_modelo_canonico/03_union/Canonico_Union_Trabajo.csv"
salida = "../03_perfilado/resumen_perfilado.csv"

df = pd.read_csv(archivo, dtype=str, keep_default_na=False)

columnas = ["Fuente_origen", "indice", "Titulo", "Año", "Autor_norm",
            "Afiliacion1", "Afiliacion2", "ISBN", "ISSN", "Doi", "URL",
            "Area", "SubArea", "Keywords", "Abstract"]

assert df.shape == (2582, 15), f"Dimensiones inesperadas: {df.shape}"
assert df.columns.tolist() == columnas, "Columnas distintas del esquema esperado"

print(f"Filas: {len(df):,} | Columnas: {len(df.columns)}")
df.head()

## 1. Registros por fuente

In [ ]:
fuentes = df["Fuente_origen"].value_counts()
print("Valores distintos de Fuente_origen:", len(fuentes))
display(fuentes.to_frame("Filas"))

fuentes.head(12).sort_values().plot.barh(figsize=(9, 5))
plt.title("Registros por fuente (12 valores más frecuentes)")
plt.xlabel("Filas")
plt.ylabel("Fuente_origen")
plt.tight_layout()
plt.show()

## 2. Completitud de los campos

In [ ]:
vacios = df.apply(lambda c: c.str.strip().eq(""))

perfil = pd.DataFrame({
    "Total": len(df),
    "Vacios": vacios.sum(),
    "Completitud": ((~vacios).mean() * 100).round(2),
    "Unicos": df.mask(vacios).nunique()
})
perfil = perfil.rename_axis("Columna").reset_index()
perfil["No_vacios"] = perfil["Total"] - perfil["Vacios"]
perfil = perfil.sort_values("Completitud").reset_index(drop=True)

display(perfil[["Columna", "Total", "No_vacios", "Vacios", "Completitud", "Unicos"]])
print(f"Completitud promedio: {perfil['Completitud'].mean():.2f}%")

ax = perfil.set_index("Columna")["Completitud"].plot.barh(figsize=(9, 6))
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3, fontsize=9)
plt.title("Completitud por columna")
plt.xlabel("Porcentaje de valores no vacíos")
plt.xlim(0, 112)
plt.xticks(range(0, 101, 20))
plt.ylabel("")
plt.tight_layout()
plt.show()

perfil[["Columna", "Total", "Vacios", "Completitud", "Unicos"]].to_csv(
    salida, index=False, encoding="utf-8-sig"
)

## 3. Año y áreas

Las frecuencias corresponden a filas de la unión, no a publicaciones únicas.

In [ ]:
for campo in ["Año", "Area"]:
    frecuencias = df[campo].str.strip().replace("", "[Vacío]").value_counts()
    display(frecuencias.to_frame("Filas"))
    frecuencias.plot.bar(figsize=(7, 3.5), rot=0)
    plt.title(f"Registros por {campo}")
    plt.ylabel("Filas")
    plt.tight_layout()
    plt.show()

display(df["SubArea"].str.strip().replace("", "[Vacío]").value_counts().to_frame("Filas"))

anio = df["Año"].str.strip()
anio_anomalo = anio.ne("") & ~anio.str.fullmatch(r"\d{4}")
print("Años con formato distinto de cuatro dígitos:", anio_anomalo.sum())

## 4. Autores

El número de nombres separados por punto y coma es aproximado. La decodificación HTML solo se usa para el conteo.

In [ ]:
autor = df["Autor_norm"].map(unescape)
partes = autor.str.split(";")
cantidad_autores = partes.map(lambda nombres: sum(bool(n.strip()) for n in nombres))

nombres = partes.explode().str.strip()
nombres = nombres.str.replace(r"\s*\([^)]*\)\s*$", "", regex=True).str.strip()
abreviados = nombres.str.contains(
    r"^(?:[A-ZÁÉÍÓÚÑ]\.\s*){2,}|,\s*(?:[A-ZÁÉÍÓÚÑ]\.?\s*){1,5}$"
).groupby(level=0).any()

numerados = df["Autor_norm"].str.contains(r"\(\s*\d+(?:\s*,\s*\d+)*\s*\)")
ids_largos = df["Autor_norm"].str.contains(r"\(\s*\d{7,12}\s*\)")
html_numerico = df["Autor_norm"].str.contains(r"&#(?:x[0-9A-Fa-f]+|\d+);")
separadores = autor.str.contains(r"^\s*;|;\s*$|;;")

display(pd.Series({
    "Filas con un autor": cantidad_autores.eq(1).sum(),
    "Filas con varios autores": cantidad_autores.gt(1).sum(),
    "Máximo aproximado de autores por fila": cantidad_autores.max(),
    "Filas con referencias numéricas": numerados.sum(),
    "Filas con posibles identificadores largos": ids_largos.sum(),
    "Filas con entidades HTML numéricas": html_numerico.sum(),
    "Filas con nombres abreviados": abreviados.sum(),
    "Filas con separadores mal formados": separadores.sum()
}, name="Cantidad").to_frame())

display(df.loc[ids_largos, "Fuente_origen"].value_counts().to_frame("Filas con posibles ID"))

rangos = pd.cut(cantidad_autores, bins=[-1, 0, 1, 5, 20, float("inf")],
                labels=["0", "1", "2–5", "6–20", "Más de 20"])
rangos.value_counts(sort=False).plot.bar(figsize=(7, 3.5), rot=0)
plt.title("Cantidad aproximada de autores por fila")
plt.xlabel("Autores")
plt.ylabel("Filas")
plt.tight_layout()
plt.show()

## 5. Afiliaciones

Se identifican menciones de la UNAM en los dos campos; esto no confirma la afiliación de cada autor.

In [ ]:
patron_unam = (r"(?<!\w)u\.?\s*n\.?\s*a\.?\s*m\.?(?!\w)|"
               r"universidad nacional autonoma de mexico|"
               r"national autonomous university of mexico")

def menciona_unam(columna):
    texto = columna.str.normalize("NFKD").str.lower()
    texto = texto.str.replace(r"[\u0300-\u036f]", "", regex=True)
    return texto.str.contains(patron_unam)

unam1 = menciona_unam(df["Afiliacion1"])
unam2 = menciona_unam(df["Afiliacion2"])

display(pd.Series({
    "Mención en Afiliacion1": unam1.sum(),
    "Mención en Afiliacion2": unam2.sum(),
    "Mención en al menos un campo": (unam1 | unam2).sum(),
    "Referencias numéricas en Afiliacion1": df["Afiliacion1"].str.contains(r"\(\d+\)").sum(),
    "Referencias numéricas en Afiliacion2": df["Afiliacion2"].str.contains(r"\(\d+\)").sum()
}, name="Filas").to_frame())

menciones = pd.Series({
    "Sin mención": (~(unam1 | unam2)).sum(),
    "En un campo": (unam1 ^ unam2).sum(),
    "En ambos campos": (unam1 & unam2).sum()
}, name="Filas")
display(menciones.to_frame())

menciones.plot.bar(figsize=(7, 3.5), rot=0)
plt.title("Menciones de la UNAM en las afiliaciones")
plt.ylabel("Filas")
plt.tight_layout()
plt.show()

## 6. Identificadores y títulos

In [ ]:
# Comparación temporal, sin cambiar los valores del archivo original.
doi = df["Doi"].str.strip().str.lower()
doi = doi.str.replace(r"^doi\s*:\s*", "", regex=True)
doi = doi.str.replace(r"^https?://(?:dx\.)?doi\.org/", "", regex=True)
doi = doi.str.replace(r"\s+", "", regex=True)

titulo = df["Titulo"].str.normalize("NFKC").str.casefold()
titulo = titulo.str.replace(r"[^\w\s]", " ", regex=True)
titulo = titulo.str.replace(r"\s+", " ", regex=True).str.strip()
anio_temp = anio.str.replace(r"^(\d{4})\.0$", r"\1", regex=True)

doi_conteos = doi[doi.ne("")].value_counts()
titulo_conteos = titulo[titulo.ne("")].value_counts()
titulo_anio = pd.DataFrame({"Titulo": titulo, "Año": anio_temp})
titulo_anio_conteos = titulo_anio[titulo.ne("")].value_counts()

doi_fuentes = df.loc[doi.ne(""), "Fuente_origen"].groupby(doi[doi.ne("")]).nunique()

display(pd.Series({
    "DOI no vacíos (temporales)": doi.ne("").sum(),
    "DOI únicos (temporales)": len(doi_conteos),
    "DOI en más de una fuente": doi_fuentes.gt(1).sum(),
    "Títulos únicos (originales, no vacíos)": df.loc[~vacios["Titulo"], "Titulo"].nunique(),
    "Títulos únicos (temporales)": len(titulo_conteos),
    "ISBN con notación científica": df["ISBN"].str.strip().str.fullmatch(r"[+-]?\d+(?:\.\d+)?[Ee][+-]?\d+").sum(),
    "ISSN con notación científica": df["ISSN"].str.strip().str.fullmatch(r"[+-]?\d+(?:\.\d+)?[Ee][+-]?\d+").sum()
}, name="Cantidad").to_frame())

## 7. Posibles registros repetidos

Se usa el DOI cuando está disponible; en su ausencia, el título y el año. Los grupos son candidatos de revisión, no duplicados confirmados.

In [ ]:
clave = pd.Series("", index=df.index)
clave.loc[titulo.ne("")] = "TY:" + titulo + "|" + anio_temp
clave.loc[doi.ne("")] = "DOI:" + doi[doi.ne("")]

grupos = df.loc[clave.ne("")].groupby(clave[clave.ne("")])
tamano = grupos.size()
varias_fuentes = grupos["Fuente_origen"].nunique().gt(1)

comparacion = pd.DataFrame({
    "Grupos repetidos": [(c > 1).sum() for c in
                        [doi_conteos, titulo_conteos, titulo_anio_conteos, tamano]],
    "Filas involucradas": [c[c > 1].sum() for c in
                          [doi_conteos, titulo_conteos, titulo_anio_conteos, tamano]]
}, index=["DOI", "Título", "Título y año", "Clave bibliográfica"])

display(comparacion)
print("Grupos bibliográficos aproximados:", len(tamano))
print("Grupos en más de una fuente:", varias_fuentes.sum())
print("Filas en grupos de varias fuentes:", tamano[varias_fuentes].sum())

comparacion["Filas involucradas"].plot.barh(figsize=(8, 3.5))
plt.title("Filas en grupos de posibles registros repetidos")
plt.xlabel("Filas")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 8. Resumen

In [ ]:
print(f"Base de entrada: {len(df):,} filas y {len(df.columns)} columnas")
print(f"Completitud promedio: {perfil['Completitud'].mean():.2f}%")
print(f"Filas con varios autores: {cantidad_autores.gt(1).sum():,}")
print(f"Filas con mención de la UNAM: {(unam1 | unam2).sum():,}")
print(f"Grupos bibliográficos aproximados: {len(tamano):,}")
print("Resultado guardado:", salida)
print("El archivo de entrada no se modificó.")